# Générateur de formes abstraites

Pipeline : **forme de base** → **modules de déformation** (activables, appliqués dans l'ordre) → **emboîtement** éventuel → **visualisation 3D** → **STL / G-code**.

Chaque module a un drapeau `actif` : passez-le à `False` pour le couper sans perdre ses réglages.
Les fichiers produits vont dans `sorties/`.

In [ ]:
from pathlib import Path
from formes import *

SORTIES = Path("sorties")
SORTIES.mkdir(exist_ok=True)
NOM = "forme-01"   # préfixe des fichiers exportés

## 1. Forme de base

Profils : `cercle`, `polygone`, `etoile`, `superellipse`, `goutte`.
`couches` et `points` règlent la finesse du calcul (pas la hauteur de couche d'impression).

In [ ]:
base = Base(
    profil="etoile",
    rayon=40,        # mm
    hauteur=110,     # mm
    cotes=6,
    creux=0.25,
    couches=150,
    points=360,
)

## 2. Modules de déformation

L'ordre de la liste est l'ordre d'application. Mettez `actif=False` pour désactiver un module.

| module | effet |
|---|---|
| `Fuseau` | évasement, rétrécissement, galbe |
| `Torsion` | rotation progressive des couches |
| `Ondulation` | vagues radiales hélicoïdales |
| `Plis` | plissé à arêtes vives |
| `Bruit` | relief organique aléatoire (graine) |
| `Pincement` | aplatissement elliptique qui tourne |
| `Courbure` | inclinaison, arc, serpentin |

In [ ]:
deformations = [
    Fuseau(echelle_haut=0.75, ventre=0.3, position_ventre=0.35, actif=True),
    Torsion(angle=120, acceleration=1.3,                         actif=True),
    Ondulation(amplitude=2.5, lobes=12, tours=1.5,               actif=True),
    Plis(profondeur=3, nombre=36,                                actif=False),
    Bruit(amplitude=3, detail=5, graine=42,                      actif=True),
    Pincement(intensite=0.25, rotation=90,                       actif=False),
    Courbure(fleche=12, direction=30,                            actif=False),
]

## 3. Emboîtement (optionnel)

- `Gigogne` : coques imbriquées, imprimables en place en une seule fois.
- `Empilement` : la forme est découpée en modules ; chacun porte un col qui s'emboîte dans le suivant (vue éclatée réglable avec `ecart_vue`).
- `Reseau` : copies en grille avec rotation/échelle progressives.

Choisissez-en un (ou `None`).

In [ ]:
emboitement = None
# emboitement = Gigogne(nombre=3, paroi=0.8, jeu=1.5, retrait_hauteur=10)
# emboitement = Empilement(nombre=3, hauteur_col=8, jeu=0.3, ecart_vue=15)
# emboitement = Reseau(colonnes=3, lignes=1, rotation_pas=20)

## 4. Génération et visualisation 3D

In [ ]:
gen = Generateur(base, deformations, emboitement)
assemblage = gen.generer()
print(gen)

fig = figure_3d(assemblage, titre=NOM)
fig.show()

### Panneau interactif (optionnel)

Une case « actif » et des curseurs par module, générés automatiquement.
Chaque changement régénère la vue : baissez `couches`/`points` de la base si c'est lent.
Les réglages modifiés ici sont conservés dans `gen` pour les exports ci-dessous.

In [ ]:
from formes.interface import panneau

panneau(gen, lambda g: figure_3d(g.generer(), titre=NOM))

## 5. Exports : visualisation HTML et STL

In [ ]:
assemblage = gen.generer()   # reprend les réglages du panneau
print("HTML :", exporter_html(figure_3d(assemblage, titre=NOM), SORTIES / f"{NOM}.html"))
print("STL  :", assemblage.exporter_stl(SORTIES / f"{NOM}.stl"))

## 6. G-code Klipper (Mainsail / Moonraker / KAMP)

Le fichier produit contient :
- **vignettes** 32×32 et 300×300 → aperçu dans Mainsail, Fluidd et KlipperScreen ;
- **métadonnées** lues par Moonraker : durée estimée, filament (m et g), nombre de couches, hauteur, températures, buse ;
- **`EXCLUDE_OBJECT_DEFINE`** avant la macro de démarrage, puis `EXCLUDE_OBJECT_START/END` autour de chaque pièce → annulation d'une pièce depuis Mainsail, maillage et purge adaptatifs **KAMP** ;
- `SET_PRINT_STATS_INFO` → « couche x / n » dans Mainsail.

Côté `printer.cfg` : il faut une section `[exclude_object]` et une macro `PRINT_START` qui accepte `BED` et `EXTRUDER` (modifiable avec `macro_debut`).
Si votre `PRINT_START` lance déjà le maillage et la purge, laissez `maillage_adaptatif=False` et `purge=None`.

Modes : `spirale` (vase, une pièce, sans couture) ou `couches` (plusieurs pièces, bascule automatique).
`firmware="marlin"` produit un G-code Marlin classique (M190/M109/G28, sans EXCLUDE_OBJECT).

In [ ]:
impression = ParametresImpression(
    firmware="klipper",
    mode="spirale",
    hauteur_couche=0.2,
    largeur_ligne=0.45,
    diametre_buse=0.4,
    diametre_filament=1.75,
    couches_fond=4,
    temp_buse=210,
    temp_plateau=60,
    vitesse=25,
    plateau=(220, 220),
    # --- Klipper / KAMP ---
    macro_debut="PRINT_START BED={temp_plateau} EXTRUDER={temp_buse}",
    macro_fin="PRINT_END",
    maillage_adaptatif=True,      # BED_MESH_CALIBRATE ADAPTIVE=1 (False si PRINT_START le fait)
    purge="LINE_PURGE",           # macro KAMP ; "interne" = ligne générée ici ; None = aucune
    exclude_object=True,
    retraction_firmware=False,    # True : G10/G11 avec [firmware_retraction]
    # --- métadonnées Mainsail ---
    filament_type="PLA",
    filament_nom="PLA Galaxy",
    filament_couleur="#E8B04A",
    vignettes=((32, 32), (300, 300)),
)

trajet = exporter_gcode(assemblage, SORTIES / f"{NOM}.gcode", impression)
for k, v in trajet.stats.items():
    print(f"{k:>10} : {v}")
for a in trajet.avertissements:
    print("⚠", a)

In [ ]:
# Aperçu de la vignette telle que Mainsail l'affichera
from IPython.display import Image, display
from formes.vignettes import png, rendu_rgba, _recadrer

display(Image(png(_recadrer(rendu_rgba(assemblage.pour_impression().pieces, 900)), 300, 300)))

In [ ]:
fig_trajet = figure_trajet(trajet, titre=f"{NOM} — trajet G-code")
exporter_html(fig_trajet, SORTIES / f"{NOM}-trajet.html")
fig_trajet.show()

## Ajouter un module

Créez une classe `@dataclass` héritant de `Deformation` (dans `formes/deformations.py`), déclarez ses réglages avec `param(défaut, min, max, pas, "aide")` et implémentez `appliquer(forme)` en ne modifiant que x et y de `forme.pts` (tableau couches × points × 3). Le panneau interactif crée ses curseurs tout seul.